# Principal Component Analysis (PCA)

PCA is an unsupervised dimensionality reduction method. It projects data onto **principal components** — orthogonal directions that capture maximum variance in descending order.

**Dataset:** Iris — 4 features reduced to 2 components for visualisation.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
import sys
sys.path.insert(0, '.')
from pca import pca as PCA

iris = load_iris()
X, y = iris.data, iris.target
feature_names = iris.feature_names
class_names   = iris.target_names
print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features")
print(f"Features: {list(feature_names)}")
print(f"Classes:  {list(class_names)}")

## Standardise the Data

PCA is scale-sensitive — we standardise so no feature dominates due to its units.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Mean (should be ~0):", X_scaled.mean(axis=0).round(4))
print("Std  (should be ~1):", X_scaled.std(axis=0).round(4))

## Fit PCA — All Components & Scree Plot

In [ ]:
model_full = PCA(n_components=None)
model_full.fit(X_scaled)

print("Explained variance ratio per component:")
for i, r in enumerate(model_full.explained_variance_ratio_):
    bar = chr(9608) * int(r * 40)
    print(f"  PC{i+1}: {r:.4f}  {bar}")
print(f"\nCumulative: {np.cumsum(model_full.explained_variance_ratio_).round(4)}")

cumvar = np.cumsum(model_full.explained_variance_ratio_)
plt.figure(figsize=(8,4))
plt.bar(range(1,5), model_full.explained_variance_ratio_, color='steelblue', alpha=0.7, label='Individual')
plt.plot(range(1,5), cumvar, 'o-', color='tomato', linewidth=2, label='Cumulative')
plt.axhline(0.95, color='gray', linestyle='--', alpha=0.6, label='95% threshold')
plt.xlabel("Principal Component")
plt.ylabel("Explained Variance Ratio")
plt.title("Scree Plot")
plt.xticks(range(1,5))
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Project to 2 Principal Components

In [ ]:
model_2d = PCA(n_components=2)
X_2d = model_2d.fit_transform(X_scaled)
print(f"Shape: {X_scaled.shape} -> {X_2d.shape}")
print(f"Variance retained: {model_2d.explained_variance_ratio_.sum()*100:.1f}%")

COLORS = ['#e63946','#457b9d','#2a9d8f']
plt.figure(figsize=(8,6))
for cls in range(3):
    mask = y == cls
    plt.scatter(X_2d[mask,0], X_2d[mask,1], color=COLORS[cls],
                label=class_names[cls], alpha=0.8, edgecolors='white', linewidths=0.3, s=55)
plt.xlabel(f"PC1 ({model_2d.explained_variance_ratio_[0]*100:.1f}% variance)")
plt.ylabel(f"PC2 ({model_2d.explained_variance_ratio_[1]*100:.1f}% variance)")
plt.title("Iris — PCA 2D Projection")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Reconstruction Error vs Components

In [ ]:
print(f"{'Components':>12} | {'Variance':>10} | {'Recon. Error':>14}")
print("-" * 44)
for n in [1, 2, 3, 4]:
    m = PCA(n_components=n)
    m.fit(X_scaled)
    err = m.reconstruction_error(X_scaled)
    var = m.explained_variance_ratio_.sum()
    print(f"{n:>12} | {var:>10.4f} | {err:>14.6f}")

## Component Loadings

Loadings show each feature's contribution to a principal component.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,4))
for i, ax in enumerate(axes):
    loadings = model_2d.components_[i]
    bar_colors = ['steelblue' if v >= 0 else 'tomato' for v in loadings]
    ax.bar(feature_names, loadings, color=bar_colors, edgecolor='white')
    ax.axhline(0, color='gray', linewidth=0.8)
    ax.set_title(f"PC{i+1} Loadings ({model_2d.explained_variance_ratio_[i]*100:.1f}% variance)")
    ax.set_ylabel("Loading")
    ax.tick_params(axis='x', rotation=15)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Key Takeaways

- Two PCs capture **~95.8%** of Iris variance — near-lossless reduction.
- *Setosa* is clearly separable; *versicolor* and *virginica* partially overlap in 2D.
- PC1 is dominated by petal length/width; PC2 by sepal dimensions.
- Lower reconstruction error = more information preserved after reduction.
